# 性能分析与计时代码：


在开发代码和创建数据处理管道的过程中，通常会面临不同实现之间的权衡。

在算法开发初期，过于关注这些问题可能是适得其反的。正如唐纳德·克努斯（Donald Knuth）所言：“我们应该忘记小效率，大约97%的时间都应如此：过早优化是一切邪恶之源。”

但一旦你的代码能够正常工作，深入研究其效率就变得有意义了。

有时检查某个命令或命令集的执行时间是很有用的；而其他时候，则需要分析一个多行过程，以确定复杂操作序列中的瓶颈所在。

IPython 提供了一系列功能，用于对代码进行计时和性能分析。

接下来，我们将讨论以下 IPython 魔法命令：

- `%time`：测量单个语句的执行时间

- `%timeit`：更准确地测量单个语句重复执行的时间

- `%prun`：使用性能分析器运行代码

- `%lprun`：使用逐行性能分析器运行代码

- `%memit`：测量单个语句的内存使用情况

- `%mprun`：使用逐行内存分析器运行代码

最后四条命令并未与 IPython 一起捆绑；要使用它们，你需要安装 `line_profiler` 和 `memory_profiler` 扩展，我们将在后后续部分中讨论这些内容。

## 计时代码片段：%timeit 和 %time

我们在 [IPython 魔法命令](01.03-Magic-Commands.ipynb) 的魔法函数介绍中看到了 `%timeit` 行魔法和 `%%timeit` 单元魔法；这些可以用于测量代码片段的重复执行时间。

In [13]:
%timeit sum(range(100))

191 ns ± 2.18 ns per loop (mean ± std. dev. of 7 runs, 10,000,000 loops each)


请注意，由于此操作非常快速，`%timeit` 会自动进行大量重复测试。

对于较慢的命令，`%timeit` 将自动调整并执行更少的重复测试。

In [14]:
%%timeit
total = 0
for i in range(1000):
    for j in range(1000):
        total += i * (-1) ** j

52.3 ms ± 895 μs per loop (mean ± std. dev. of 7 runs, 10 loops each)


有时候，重复一个操作并不是最佳选择。

例如，如果我们有一个想要排序的列表，重复操作可能会误导我们；对已排序列表进行排序比对未排序列表进行排序快得多，因此这种重复会扭曲结果。

In [15]:
import random
L = [random.random() for i in range(100000)]
%timeit L.sort()

201 μs ± 4.77 μs per loop (mean ± std. dev. of 7 runs, 10,000 loops each)


为了这个目的，`%time` 魔法函数可能是更好的选择。对于运行时间较长的命令，它也是一个不错的选择，因为短暂的系统相关延迟不太可能影响结果。

现在让我们来计时对一个未排序列表和一个已排序列表进行排序：

In [16]:
import random
L = [random.random() for i in range(100000)]
print("sorting an unsorted list:")
%time L.sort()

sorting an unsorted list:
CPU times: user 8.91 ms, sys: 0 ns, total: 8.91 ms
Wall time: 9.17 ms


In [17]:
print("sorting an already sorted list:")
%time L.sort()

sorting an already sorted list:
CPU times: user 247 μs, sys: 0 ns, total: 247 μs
Wall time: 249 μs


注意预排序列表的排序速度有多快，但也要注意，即使对于预排序列表，使用 `%time` 的计时所需时间比 `%timeit` 要长得多！

这是因为 `%timeit` 在后台进行了一些巧妙的操作，以防止系统调用干扰计时。

例如，它会阻止未使用的 Python 对象（称为 *垃圾回收*）的清理，这可能会影响计时结果。

因此，通常情况下，`%timeit` 的结果明显比 `%time` 更快。

对于 `%time` 和 `%timeit`，使用 `%%` 单元魔法语法可以对多行脚本进行计时。

In [18]:
%%time
total = 0
for i in range(1000):
    for j in range(1000):
        total += i * (-1) ** j

CPU times: user 71.1 ms, sys: 0 ns, total: 71.1 ms
Wall time: 70.9 ms


有关 `%time` 和 `%timeit` 及其可用选项的更多信息，请使用 IPython 帮助功能（例如，在 IPython 提示符下输入 `%time?`）。

## Profiling Full Scripts: %prun

一个程序由多个单独的语句组成，有时在上下文中对这些语句进行计时比单独计时更为重要。

Python包含一个内置的代码分析器（您可以在Python文档中阅读相关内容），而IPython则提供了一种更方便的方法来使用这个分析器，即魔法函数`%prun`。

作为示例，我们将定义一个执行一些计算的简单函数：




In [19]:
def sum_of_lists(N):
    total = 0
    for i in range(5):
        L = [j ^ (j >> i) for j in range(N)]
        total += sum(L)
    return total

现在我们可以通过函数调用来使用 `%prun`，以查看性能分析结果：

In [20]:
%prun sum_of_lists(1000000)

         564 function calls (558 primitive calls) in 0.237 seconds

   Ordered by: internal time

   ncalls  tottime  percall  cumtime  percall filename:lineno(function)
        1    0.120    0.120    0.200    0.200 <string>:1(<module>)
        1    0.056    0.056    0.066    0.066 3519952779.py:1(sum_of_lists)
        5    0.027    0.005    0.027    0.005 {built-in method builtins.sum}
        2    0.011    0.006    0.015    0.007 {method '__exit__' of 'sqlite3.Connection' objects}
        1    0.006    0.006    0.026    0.026 history.py:99(only_when_enabled)
        1    0.006    0.006    0.006    0.006 {method 'execute' of 'sqlite3.Connection' objects}
       14    0.006    0.000    0.006    0.000 socket.py:623(send)
      2/1    0.004    0.002    0.200    0.200 {built-in method builtins.exec}
        1    0.000    0.000    0.000    0.000 {method 'disable' of '_lsprof.Profiler' objects}
        4    0.000    0.000    0.000    0.000 attrsettr.py:66(_get_attr_opt)
        2    0.000  

结果是一个表格，按每个函数调用的总时间排序，显示了执行过程中花费最多时间的地方。在这种情况下，大部分执行时间都在 `sum_of_lists` 内部的列表推导中。

从这里开始，我们可以考虑对算法进行哪些改进，以提高其性能。

有关 `%prun` 及其可用选项的更多信息，请使用 IPython 帮助功能（即，在 IPython 提示符下输入 `%prun?`）。

## Line-by-Line Profiling with %lprun

The function-by-function profiling of `%prun` is useful, but sometimes it's more convenient to have a line-by-line profile report.
This is not built into Python or IPython, but there is a `line_profiler` package available for installation that can do this.
Start by using Python's packaging tool, `pip`, to install the `line_profiler` package:

```
$ pip install line_profiler
```

Next, you can use IPython to load the `line_profiler` IPython extension, offered as part of this package:

In [21]:
%load_ext line_profiler

The line_profiler extension is already loaded. To reload it, use:
  %reload_ext line_profiler


Now the `%lprun` command will do a line-by-line profiling of any function. In this case, we need to tell it explicitly which functions we're interested in profiling:

In [22]:
%lprun -f sum_of_lists sum_of_lists(5000)

Timer unit: 1e-09 s

Total time: 0.00102545 s
File: /tmp/ipykernel_1465689/3519952779.py
Function: sum_of_lists at line 1

Line #      Hits         Time  Per Hit   % Time  Line Contents
     1                                           def sum_of_lists(N):
     2         1        413.0    413.0      0.0      total = 0
     3         6       1194.0    199.0      0.1      for i in range(5):
     4         5     929909.0 185981.8     90.7          L = [j ^ (j >> i) for j in range(N)]
     5         5      93357.0  18671.4      9.1          total += sum(L)
     6         1        578.0    578.0      0.1      return total

The information at the top gives us the key to reading the results: the time is reported in microseconds, and we can see where the program is spending the most time.
At this point, we may be able to use this information to modify aspects of the script and make it perform better for our desired use case.

For more information on `%lprun`, as well as its available options, use the IPython help functionality (i.e., type `%lprun?` at the IPython prompt).

## Profiling Memory Use: %memit and %mprun

Another aspect of profiling is the amount of memory an operation uses.
This can be evaluated with another IPython extension, the `memory_profiler`.
As with the `line_profiler`, we start by `pip`-installing the extension:

```
$ pip install memory_profiler
```

Then we can use IPython to load it:

In [24]:
!pip install memory_profiler

In [25]:
%load_ext memory_profiler

The memory profiler extension contains two useful magic functions: `%memit` (which offers a memory-measuring equivalent of `%timeit`) and `%mprun` (which offers a memory-measuring equivalent of `%lprun`).
The `%memit` magic function can be used rather simply:

In [26]:
%memit sum_of_lists(1000000)

peak memory: 166.41 MiB, increment: 73.07 MiB


我们看到这个函数大约使用了140 MB的内存。

为了逐行描述内存使用情况，我们可以使用`%mprun`魔法函数。

不幸的是，这仅适用于在单独模块中定义的函数，而不是在笔记本中。因此，我们将首先使用`%%file`单元魔法来创建一个名为`mprun_demo.py`的简单模块，该模块包含我们的`sum_of_lists`函数，并增加一项内容，以使我们的内存分析结果更加清晰。

In [27]:
%%file mprun_demo.py
def sum_of_lists(N):
    total = 0
    for i in range(5):
        L = [j ^ (j >> i) for j in range(N)]
        total += sum(L)
        del L # remove reference to L
    return total

Writing mprun_demo.py


我们现在可以导入此函数的新版本，并运行内存行分析器：

In [28]:
from mprun_demo import sum_of_lists
%mprun -f sum_of_lists sum_of_lists(1000000)

Filename: /home/jovyan/work/PythonDataScienceHandbook/notebooks/mprun_demo.py

Line #    Mem usage    Increment  Occurrences   Line Contents
     1     93.5 MiB     93.5 MiB           1   def sum_of_lists(N):
     2     93.5 MiB      0.0 MiB           1       total = 0
     3    100.9 MiB     -0.4 MiB           6       for i in range(5):
     4    129.9 MiB -45822755.1 MiB     5000005           L = [j ^ (j >> i) for j in range(N)]
     5    129.9 MiB      0.0 MiB           5           total += sum(L)
     6    100.9 MiB   -116.0 MiB           5           del L # remove reference to L
     7    100.9 MiB     -0.1 MiB           1       return total

在这里，`Increment` 列告诉我们每一行对总内存预算的影响：注意，当我们创建和删除列表 `L` 时，内存使用量增加了大约 30 MB。

这还不包括 Python 解释器本身的背景内存使用。

有关 `%memit` 和 `%mprun` 的更多信息以及它们可用的选项，请使用 IPython 帮助功能（例如，在 IPython 提示符下输入 `%memit?`）。